# Baseline-модели прогнозирования спроса

Сравнение базовых моделе

Метрики: **RMSE**, **MAE**, **R2**. Сравнительная таблица

In [112]:
import os
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, \
    mean_absolute_error, r2_score

from sklearn.preprocessing import StandardScaler

In [2]:
os.chdir("..")

## Конфигурация

In [ ]:
DATA_PATH = "data/interim/monthly_sales.parquet"
TARGET_COL = "item_cnt_month"
TIME_COL = "date_block_num"

TEST_MONTH_TRESHOLD = 24 # для теста берем 3 последних месяца

FEATURE_COLS = ["lag_1", "lag_2", "lag_3", "month"]

## Загрузка данных

In [60]:
df = pd.read_parquet(DATA_PATH)
df = df.sort_values(["shop_id", TIME_COL, "item_id"], ascending=[False, False, True])

print(f"Строк: {len(df):,}")
print(f"Месяцы: {df[TIME_COL].min()} — {df[TIME_COL].max()}")
df.tail()

Строк: 1,597,126
Месяцы: 0 — 33


,date_block_num,year,month,shop_id,item_id,item_cnt_month,avg_price,item_name,item_category_id,item_category_name,main_category,super_category,shop_name
723,0,2013,1,2,22088,1.0,79.0,Элемент питания DURACELL LR06-BC2,83,Элементы питания,Элементы питания,Прочее,"Адыгея ТЦ ""Мега"""
724,0,2013,1,2,22091,1.0,109.0,Элемент питания DURACELL TURBO LR 03 2*BL,83,Элементы питания,Элементы питания,Прочее,"Адыгея ТЦ ""Мега"""
725,0,2013,1,2,22092,4.0,109.0,Элемент питания DURACELL TURBO LR6 2*BL,83,Элементы питания,Элементы питания,Прочее,"Адыгея ТЦ ""Мега"""
726,0,2013,1,2,22095,2.0,60.0,Элемент питания GP 15AU (LR6) - Блистер 2 шт U...,83,Элементы питания,Элементы питания,Прочее,"Адыгея ТЦ ""Мега"""
727,0,2013,1,2,22168,1.0,799.0,Яйцо для Little Inu,62,"Подарки - Гаджеты, роботы, спорт",Подарки,Подарки,"Адыгея ТЦ ""Мега"""


### Просмотр пропусков в date_block_num для товаров. Если пропуск есть - товар за данный месяц не продавался

In [38]:
all_shops = df['shop_id'].unique()
all_items = df['item_id'].unique()
all_months = df['date_block_num'].unique()

# декартово произведение
from itertools import product
all_combinations = pd.DataFrame(
    list(product(all_shops, all_items, all_months)),
    columns=['shop_id', 'item_id', 'date_block_num']
)

existing = df[['shop_id', 'item_id', 'date_block_num']].copy()
existing['exists'] = 1

full_df = all_combinations.merge(existing, on=['shop_id', 'item_id', 'date_block_num'], how='left')
missing = full_df[full_df['exists'].isna()]

print(f"Всего возможных комбинаций: {len(all_combinations)}")
print(f"Присутствует в данных: {len(existing)}")
print(f"Отсутствует: {len(missing)}")
print("\nПример отсутствующих комбинаций:")
print(missing.head(10))

Всего возможных комбинаций: 42802260
Присутствует в данных: 1597126
Отсутствует: 41205134

Пример отсутствующих комбинаций:
    shop_id  item_id  date_block_num  exists
1        59       31              32     NaN
3        59       31              30     NaN
4        59       31              29     NaN
5        59       31              28     NaN
6        59       31              27     NaN
7        59       31              26     NaN
8        59       31              25     NaN
9        59       31              24     NaN
11       59       31              22     NaN
12       59       31              21     NaN


In [61]:
import pandas as pd

# Справочники 
month_mapping = df[['date_block_num', 'year', 'month']].drop_duplicates().set_index('date_block_num')
items_static = df[['item_id', 'item_name', 'item_category_id', 
                   'item_category_name', 'main_category', 'super_category']].drop_duplicates()
shops_static = df[['shop_id', 'shop_name']].drop_duplicates()

# Функция заполнения
def fill_shop(group):
    items = group['item_id'].unique()
    months = range(group['date_block_num'].min(), group['date_block_num'].max() + 1)
    idx = pd.MultiIndex.from_product([items, months], names=['item_id', 'date_block_num'])
    temp = group[['item_id', 'date_block_num', 'item_cnt_month', 'avg_price']].set_index(['item_id', 'date_block_num'])
    full = temp.reindex(idx).reset_index()
    full['shop_id'] = group['shop_id'].iloc[0]
    full['item_cnt_month'] = full['item_cnt_month'].fillna(0)
    full = full.sort_values(['shop_id', 'item_id', 'date_block_num'])
    full['avg_price'] = full.groupby(['shop_id', 'item_id'])['avg_price'].ffill()
    return full

filled_dfs = []
for shop_id, group in df.groupby('shop_id'):
    filled_dfs.append(fill_shop(group))

full_df = pd.concat(filled_dfs, ignore_index=True)

full_df = full_df.merge(month_mapping, on='date_block_num', how='left')
full_df = full_df.merge(items_static, on='item_id', how='left')
full_df = full_df.merge(shops_static, on='shop_id', how='left')

full_df = full_df[df.columns]

In [62]:
full_df.head()

,date_block_num,year,month,shop_id,item_id,item_cnt_month,avg_price,item_name,item_category_id,item_category_name,main_category,super_category,shop_name
0,0,2013,1,2,27,1.0,2499.0,"007 Legends [PS3, русская версия]",19,Игры - PS3,Игры,Игры,"Адыгея ТЦ ""Мега"""
1,1,2013,2,2,27,0.0,2499.0,"007 Legends [PS3, русская версия]",19,Игры - PS3,Игры,Игры,"Адыгея ТЦ ""Мега"""
2,2,2013,3,2,27,0.0,2499.0,"007 Legends [PS3, русская версия]",19,Игры - PS3,Игры,Игры,"Адыгея ТЦ ""Мега"""
3,3,2013,4,2,27,0.0,2499.0,"007 Legends [PS3, русская версия]",19,Игры - PS3,Игры,Игры,"Адыгея ТЦ ""Мега"""
4,4,2013,5,2,27,0.0,2499.0,"007 Legends [PS3, русская версия]",19,Игры - PS3,Игры,Игры,"Адыгея ТЦ ""Мега"""


In [63]:
full_df.isna().sum()

date_block_num              0
year                        0
month                       0
shop_id                     0
item_id                     0
item_cnt_month              0
avg_price             4095129
item_name                   0
item_category_id            0
item_category_name          0
main_category               0
super_category              0
shop_name                   0
dtype: int64

In [64]:
memory_usage = full_df.memory_usage(deep=True).sum() / 1024**3
print(f"DataFrame занимает {memory_usage:.2f} ГБ")

DataFrame занимает 9.68 ГБ


In [65]:
full_df[(full_df['shop_id'] == 2) & (full_df['item_id'] == 31)].tail()

,date_block_num,year,month,shop_id,item_id,item_cnt_month,avg_price,item_name,item_category_id,item_category_name,main_category,super_category,shop_name
97,29,2015,6,2,31,0.0,415.92,007: КООРДИНАТЫ «СКАЙФОЛЛ» (BD),37,Кино - Blu-Ray,Кино,Кино,"Адыгея ТЦ ""Мега"""
98,30,2015,7,2,31,0.0,415.92,007: КООРДИНАТЫ «СКАЙФОЛЛ» (BD),37,Кино - Blu-Ray,Кино,Кино,"Адыгея ТЦ ""Мега"""
99,31,2015,8,2,31,0.0,415.92,007: КООРДИНАТЫ «СКАЙФОЛЛ» (BD),37,Кино - Blu-Ray,Кино,Кино,"Адыгея ТЦ ""Мега"""
100,32,2015,9,2,31,0.0,415.92,007: КООРДИНАТЫ «СКАЙФОЛЛ» (BD),37,Кино - Blu-Ray,Кино,Кино,"Адыгея ТЦ ""Мега"""
101,33,2015,10,2,31,1.0,399.00,007: КООРДИНАТЫ «СКАЙФОЛЛ» (BD),37,Кино - Blu-Ray,Кино,Кино,"Адыгея ТЦ ""Мега"""


## Признаки и прогнозы rule-based baseline

In [66]:
group_key = ["shop_id", "item_id"]
sales = full_df.groupby(group_key, sort=False)[TARGET_COL]

full_df["lag_1"] = sales.shift(1)
full_df["lag_2"] = sales.shift(2)
full_df["lag_3"] = sales.shift(3)
full_df["lag_12"] = sales.shift(12)

full_df["pred_naive"] = full_df["lag_1"]
full_df["pred_seasonal"] = full_df["lag_12"]
full_df["pred_hist_avg"] = sales.transform(lambda s: s.shift(1).expanding().mean())

full_df.head()

,date_block_num,year,month,shop_id,item_id,item_cnt_month,avg_price,item_name,item_category_id,item_category_name,main_category,super_category,shop_name,lag_1,lag_2,lag_3,lag_12,pred_naive,pred_seasonal,pred_hist_avg
0,0,2013,1,2,27,1.0,2499.0,"007 Legends [PS3, русская версия]",19,Игры - PS3,Игры,Игры,"Адыгея ТЦ ""Мега""",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2013,2,2,27,0.0,2499.0,"007 Legends [PS3, русская версия]",19,Игры - PS3,Игры,Игры,"Адыгея ТЦ ""Мега""",1.0,NaN,NaN,NaN,1.0,NaN,1.000000
2,2,2013,3,2,27,0.0,2499.0,"007 Legends [PS3, русская версия]",19,Игры - PS3,Игры,Игры,"Адыгея ТЦ ""Мега""",0.0,1.0,NaN,NaN,0.0,NaN,0.500000
3,3,2013,4,2,27,0.0,2499.0,"007 Legends [PS3, русская версия]",19,Игры - PS3,Игры,Игры,"Адыгея ТЦ ""Мега""",0.0,0.0,1.0,NaN,0.0,NaN,0.333333
4,4,2013,5,2,27,0.0,2499.0,"007 Legends [PS3, русская версия]",19,Игры - PS3,Игры,Игры,"Адыгея ТЦ ""Мега""",0.0,0.0,0.0,NaN,0.0,NaN,0.250000


In [ ]:
full_df['lag_1'].fillna(0, inplace=True)
full_df['lag_2'].fillna(0, inplace=True)
full_df['lag_3'].fillna(0, inplace=True)
full_df['lag_12'].fillna(0, inplace=True)

## Разделение на train / test

In [92]:
full_df = full_df.sort_values(['shop_id', 'item_id', 'date_block_num'])

train = full_df[full_df['date_block_num'] <= TEST_MONTH_TRESHOLD].copy()
train = full_df[full_df['date_block_num'] >= 13].copy() # для lag 12
test = full_df[full_df['date_block_num'] > TEST_MONTH_TRESHOLD].copy()

train.shape, test.shape

((8120355, 20), (2514892, 20))

In [93]:
train['item_id'].nunique()

21679

In [94]:
test['item_id'].nunique()

21632

In [95]:
test_items = set(test['item_id'].unique())

train_filtered = train[train['item_id'].isin(test_items)].copy()

print(f"Размер train до: {len(train)}")
print(f"Размер train после: {len(train_filtered)}")

Размер train до: 8120355
Размер train после: 8119818


## Baseline Метрики

### Предсказание за предыдущий месяц

In [101]:
last_month_mae_train = mean_absolute_error(train['item_cnt_month'], train['lag_1'])
last_month_mae_test = mean_absolute_error(test['item_cnt_month'], test['lag_1'])

last_month_rmse_train = root_mean_squared_error(train['item_cnt_month'], train['lag_1'])
last_month_rmse_test = root_mean_squared_error(test['item_cnt_month'], test['lag_1'])

last_month_r2_train = r2_score(train['item_cnt_month'], train['lag_1'])
last_month_r2_test = r2_score(test['item_cnt_month'], test['lag_1'])

In [102]:
data = {
    "Metric": ["MAE", "RMSE", "R2"],
    "Train": [last_month_mae_train, last_month_rmse_train, last_month_r2_train],
    "Test": [last_month_mae_test, last_month_rmse_test, last_month_r2_test],
}

metrics_df = pd.DataFrame(data)

metrics_df

,Metric,Train,Test
0,MAE,0.257877,0.210485
1,RMSE,1.237949,1.111419
2,R2,0.401604,0.358140


### Предсказание с годовым лагом

In [103]:
last_year_mae_train = mean_absolute_error(train['item_cnt_month'], train['lag_12'])
last_year_mae_test = mean_absolute_error(test['item_cnt_month'], test['lag_12'])

last_year_rmse_train = root_mean_squared_error(train['item_cnt_month'], train['lag_12'])
last_year_rmse_test = root_mean_squared_error(test['item_cnt_month'], test['lag_12'])

last_year_r2_train = r2_score(train['item_cnt_month'], train['lag_12'])
last_year_r2_test = r2_score(test['item_cnt_month'], test['lag_12'])

In [ ]:
data_2 = {
    "Metric": ["MAE", "RMSE", "R2"],
    "Train": [last_year_mae_train, last_year_rmse_train, last_year_r2_train],
    "Test": [last_year_mae_test, last_year_rmse_test, last_year_r2_test],
}

metrics_df_2 = pd.DataFrame(data_2)

metrics_df_2

,Metric,Train,Test
0,MAE,0.410586,0.328850
1,RMSE,1.902666,1.592961
2,R2,-0.413541,-0.318545


## Линейная регрессия

In [113]:
X_train = train[FEATURE_COLS]
X_test = test[FEATURE_COLS]

y_train = train[TARGET_COL]
y_test = test[TARGET_COL]


In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [117]:
linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

train_pred = linear_model.predict(X_train)
test_pred = linear_model.predict(X_test)

In [ ]:
linear_model.fit(train)

In [119]:
regress_mae_train = mean_absolute_error(train['item_cnt_month'], train_pred)
regress_mae_test = mean_absolute_error(test['item_cnt_month'], test_pred)

regress_rmse_train = root_mean_squared_error(train['item_cnt_month'], train_pred)
regress_rmse_test = root_mean_squared_error(test['item_cnt_month'], test_pred)

regress_r2_train = r2_score(train['item_cnt_month'], train_pred)
regress_r2_test = r2_score(test['item_cnt_month'], test_pred)

In [121]:
data_3 = {
    "Metric": ["MAE", "RMSE", "R2"],
    "Train": [regress_mae_train, regress_rmse_train, regress_r2_train],
    "Test": [regress_mae_test, regress_rmse_test, regress_r2_test],
}

metrics_df_3 = pd.DataFrame(data_3)

metrics_df_3

,Metric,Train,Test
0,MAE,0.266136,0.229127
1,RMSE,1.095861,0.979696
2,R2,0.531085,0.501267


In [124]:
print(f"Параметры линейной модели: {linear_model.coef_}\
      \nсмещение линейной модели: {linear_model.intercept_}")

Параметры линейной модели: [0.82147192 0.22559902 0.23612223 0.03190215]      
смещение линейной модели: 0.22636461090678917


## Сравнительная таблица

In [134]:
metrics_df['Model'] = 'Naive (lag_1)'
metrics_df_2['Model'] = 'Seasonal (lag_12)'
metrics_df_3['Model'] = 'Linear Regression'

summary = pd.concat([metrics_df, metrics_df_2, metrics_df_3], ignore_index=True)

summary_pivot = summary.pivot(index='Model', columns='Metric', values=['Train', 'Test'])

summary_pivot

Train                          Test                    
Metric                  MAE        R2      RMSE       MAE        R2      RMSE
Model                                                                        
Linear Regression  0.266136  0.531085  1.095861  0.229127  0.501267  0.979696
Naive (lag_1)      0.257877  0.401604  1.237949  0.210485  0.358140  1.111419
Seasonal (lag_12)  0.410586 -0.413541  1.902666  0.328850 -0.318545  1.592961

In [135]:
summary_pivot.to_csv("artifacts/tables/baseline_results.csv")